# Silver Layer — Clean & Risk-Flag Financial Data
Validate transactions, deduplicate accounts, derive fraud risk scores and credit utilisation bands.

In [ ]:
from pyspark.sql.functions import (
    col, when, lit, to_timestamp, to_date, date_format,
    hour, row_number, current_timestamp, log as spark_log
)
from pyspark.sql.window import Window

In [ ]:
# Pass-through customers — clean dimension
df_cust = spark.read.format('delta').table('bronze_customers')
df_cust = df_cust.withColumn('silver_timestamp', current_timestamp())
df_cust.write.mode('overwrite').format('delta').saveAsTable('silver_customers')
print(f'Silver customers: {df_cust.count()} rows')

In [ ]:
# Clean accounts
df_acct = spark.read.format('delta').table('bronze_accounts')

w = Window.partitionBy('account_id').orderBy(col('ingestion_timestamp').desc())
df_acct = (
    df_acct
    .withColumn('_rn', row_number().over(w))
    .filter(col('_rn') == 1)
    .drop('_rn')
)

df_acct = (
    df_acct
    .withColumn('balance', col('balance').cast('double'))
    .withColumn('credit_limit', col('credit_limit').cast('double'))
    .withColumn('credit_utilisation_pct', col('credit_utilisation_pct').cast('double'))
    .withColumn('open_date', to_date('open_date'))
    .filter(col('balance') >= 0)
    # Credit utilisation band
    .withColumn('utilisation_band',
        when(col('credit_limit') == 0, 'N/A')
        .when(col('credit_utilisation_pct') <= 30, 'Low (≤30%)')
        .when(col('credit_utilisation_pct') <= 60, 'Medium (31-60%)')
        .when(col('credit_utilisation_pct') <= 90, 'High (61-90%)')
        .otherwise('Very High (>90%)'))
    .withColumn('silver_timestamp', current_timestamp())
)

df_acct.write.mode('overwrite').format('delta').saveAsTable('silver_accounts')
print(f'Silver accounts: {df_acct.count()} rows')

In [ ]:
# Clean transactions + derive fraud risk indicators
df_txn = spark.read.format('delta').table('bronze_transactions')

w2 = Window.partitionBy('transaction_id').orderBy(col('ingestion_timestamp').desc())
df_txn = (
    df_txn
    .withColumn('_rn', row_number().over(w2))
    .filter(col('_rn') == 1)
    .drop('_rn')
)

df_txn = (
    df_txn
    .withColumn('transaction_date', to_timestamp('transaction_date'))
    .withColumn('amount', col('amount').cast('double'))
    .withColumn('is_flagged_fraud', col('is_flagged_fraud').cast('boolean'))
    .filter(col('transaction_date').isNotNull())
    .filter(col('amount') > 0)
    # Derived columns
    .withColumn('transaction_date_only', date_format('transaction_date', 'yyyy-MM-dd'))
    .withColumn('transaction_hour', hour('transaction_date'))
    .withColumn('is_night_transaction',
        (hour('transaction_date') >= 22) | (hour('transaction_date') < 6))
    .withColumn('is_international', col('country') != 'UK')
    .withColumn('is_high_value', col('amount') > 5000)
    # Simple risk score: combine signals
    .withColumn('risk_score',
        (col('is_flagged_fraud').cast('int') * 40) +
        (col('is_night_transaction').cast('int') * 15) +
        (col('is_international').cast('int') * 20) +
        (col('is_high_value').cast('int') * 25))
    .withColumn('risk_band',
        when(col('risk_score') == 0,  'Low')
        .when(col('risk_score') <= 20, 'Medium')
        .when(col('risk_score') <= 50, 'High')
        .otherwise('Critical'))
    .withColumn('silver_timestamp', current_timestamp())
)

df_txn.write.mode('overwrite').format('delta').saveAsTable('silver_transactions')
print(f'Silver transactions: {df_txn.count()} rows')